In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import RandomizedSearchCV

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


In [2]:
BASE_DIR = Path.cwd().parent   # se estiveres em notebooks/
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"

In [3]:
df = pd.read_csv(DATA_DIR / "train.csv")
df.head()
dt= pd.read_csv(DATA_DIR / "test.csv")


In [4]:
features = [
    'ChiefComplaint',
    'age',
    'NeedFastExecute',
    'CriticalStatus',
    'gender',
    'StuporStatus',
    'PainGrade']

target = 'TriageGrade'

categorical_features = ['ChiefComplaint', 'gender', 'NeedFastExecute', 'CriticalStatus', 'PainGrade', 'StuporStatus']
numerical_features = ['age']

df = df[features + [target]]
dt = dt[features + [target]]

X_train = df[features]
y_train = df[target]

X_test = dt[features]
y_test = dt[target]


In [5]:
df

,ChiefComplaint,age,NeedFastExecute,CriticalStatus,gender,StuporStatus,PainGrade,TriageGrade
0,S43.0,24,2,0.0,Male,2.0,4.0,3
1,I10,88,2,0.0,Female,2.0,1.0,3
2,S09.90XA,63,2,1.0,Male,2.0,3.0,2
3,R20.2,39,2,1.0,Male,2.0,0.0,2
4,R31.9,64,2,1.0,Male,2.0,0.0,2
...,...,...,...,...,...,...,...,...
93151,T79.9,35,2,0.0,Male,0.0,0.0,3
93152,M79.60,20,2,0.0,Female,0.0,0.0,2
93153,K92.2,79,2,1.0,Female,2.0,0.0,2
93154,U07.1,31,2,1.0,Male,2.0,0.0,2


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numerical_features
        )
    ]
)


In [7]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

param_dist = {
    "classifier__n_estimators": range(10, 51, 5), # 10 a 50, de 5 em 5
    "classifier__max_depth": range(1, 21),
    "classifier__criterion": ["gini", "entropy"]
}

search1 = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=42
)

search1.fit(X_train, y_train)
print("Best parameters:", search1.best_params_)


Best parameters: {'classifier__n_estimators': 45, 'classifier__max_depth': 14, 'classifier__criterion': 'gini'}


In [8]:
rf_pipeline_best = search1.best_estimator_

y_pred = rf_pipeline_best.predict(X_test)


In [9]:

print("RANDOM FOREST")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


RANDOM FOREST
Accuracy: 0.8911875523304493
              precision    recall  f1-score   support

           1       1.00      1.00      1.00      2058
           2       1.00      0.98      0.99     16191
           3       0.82      0.73      0.77      6864
           4       0.58      0.75      0.66      3548
           5       0.00      0.00      0.00         3

    accuracy                           0.89     28664
   macro avg       0.68      0.69      0.68     28664
weighted avg       0.90      0.89      0.90     28664



Trying to make gender into a numerical feature

In [10]:
import os

os.makedirs("models", exist_ok=True)
joblib.dump({
    "model": rf_pipeline_best,
    "features": features
},"models/random_forest_new.joblib")



['models/random_forest_new.joblib']

## Mesmo modelo mas com pesos


In [11]:
class_weights = {
    1: 1,
    2: 2,
    3: 10,
    4: 50,
    5: 100
}


rf_pipeline2 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight=class_weights,
        n_jobs=-1
    ))
])

search2 = RandomizedSearchCV(
    rf_pipeline2,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=42
)

search2.fit(X_train, y_train)

print("Best parameters:", search2.best_params_)

rf_pipeline2_best = search2.best_estimator_

y_pred_2 = rf_pipeline2_best.predict(X_test)


print("RANDOM FOREST")
print("Accuracy:", accuracy_score(y_test, y_pred_2))
print(classification_report(y_test, y_pred_2, zero_division=0))

import os

os.makedirs("models", exist_ok=True)
joblib.dump({
    "model": rf_pipeline2_best,
    "features": features
}, "models/random_forest_new_weight.joblib")



Best parameters: {'classifier__n_estimators': 45, 'classifier__max_depth': 14, 'classifier__criterion': 'gini'}
RANDOM FOREST
Accuracy: 0.7737929109684621
              precision    recall  f1-score   support

           1       1.00      1.00      1.00      2058
           2       1.00      0.96      0.98     16191
           3       0.93      0.15      0.25      6864
           4       0.35      0.99      0.52      3548
           5       0.00      0.00      0.00         3

    accuracy                           0.77     28664
   macro avg       0.66      0.62      0.55     28664
weighted avg       0.90      0.77      0.75     28664



['models/random_forest_new_weight.joblib']